In [ ]:
#模型训练
#get important features
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
# 获取己糖醇的特征数据
data = pd.read_csv(r"e:\Data\Ralstonia\newdata\final_knumber\code_data\delete_outlier\training_dataset.csv")
data1 = data.iloc[:, 1:]
data1 = data1.astype(int)

x_disaccharide = data1.iloc[:, 0:-4]
y_disaccharide = data1.loc[:, "disaccharide_metabolism"]
x_hexitol = data1.iloc[:, 0:-4]
y_hexitol = data1.loc[:, "hexitol_metabolism"]

model_disaccharide = xgb.XGBClassifier()
model_disaccharide.fit(x_disaccharide, y_disaccharide)
model_hexitol = xgb.XGBRFClassifier()
model_hexitol.fit(x_hexitol, y_hexitol)

In [ ]:
#prediction
import pandas as pd
import numpy as np

# 读取已经删除训练集重合菌株、并按训练特征顺序对齐的矩阵
new_data = pd.read_csv(
    r"E:\Data\Ralstonia\Pycharm\graduation2\new_analysis\predection\strain_1082_Knumber_aligned_nontraining.csv"
)
strain_names = new_data.iloc[:, 0].astype(str)
x_new = new_data.iloc[:, 1:].astype(int)

# 预测类别：0或1
pred_disaccharide = model_disaccharide.predict(x_new)
pred_hexitol = model_hexitol.predict(x_new)

# 预测为1的概率
disaccharide_class1_index = np.where(
    model_disaccharide.classes_ == 1
)[0][0]

hexitol_class1_index = np.where(
    model_hexitol.classes_ == 1
)[0][0]

prob_disaccharide = model_disaccharide.predict_proba(x_new)[
    :, disaccharide_class1_index
]

prob_hexitol = model_hexitol.predict_proba(x_new)[
    :, hexitol_class1_index
]

In [ ]:
prediction_result = pd.DataFrame({
    "strain": strain_names,
    "disaccharide_prediction": pred_disaccharide.astype(int),
    "disaccharide_probability": prob_disaccharide,
    "hexitol_prediction": pred_hexitol.astype(int),
    "hexitol_probability": prob_hexitol
})

output_path = (
    r"E:\Data\Ralstonia\Pycharm\graduation2"
    r"\new_analysis\predection"
    r"\prediction_results_858_strains.csv"
)

prediction_result.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)